<a href="https://colab.research.google.com/github/peperjet/algorithm/blob/main/transformers_260511.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

한국어 BERT를 이용한 영화 리뷰 분류


In [ ]:
import tensorflow as tf

print("TensorFlow version:", tf.__version__)
print("GPU 목록:", tf.config.list_physical_devices('GPU'))

if tf.config.list_physical_devices('GPU'):
    print("GPU 연결 성공")
else:
    print("GPU가 연결되지 않음")

TensorFlow version: 2.20.0
GPU 목록: []
GPU가 연결되지 않음


TPU 오류로 인하여 GPU로 연결했습니다

In [ ]:
strategy = tf.distribute.get_strategy()

print("Strategy 설정 완료")
print("장치 개수:", strategy.num_replicas_in_sync)

Strategy 설정 완료
장치 개수: 1


In [ ]:
with strategy.scope():
    # model = ...
    # model.compile(...)
    pass

create_model() = 모델 만드는 공장 함수


strategy.scope() = GPU/TPU에게 “이 모델 여기서 돌려라”라고 알려주는 영역

수업에서 TPU 쓰든 GPU 쓰든 보통 이 구조 많이 씀.

In [ ]:
def create_model():

    model = tf.keras.Sequential([
        tf.keras.layers.Dense(128, activation='relu'),
        tf.keras.layers.Dense(1, activation='sigmoid')
    ])

    model.compile(
        optimizer='adam',
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return model

In [ ]:
with strategy.scope():
    model = create_model()

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

transtormoers 모델 클래스 불러오기

In [ ]:
!pip install transformers==4.37.2 torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 129.4/129.4 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.4/8.4 MB 56.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 45.0 MB/s eta 0:00:00
  Attempting uninstall: huggingface-hub
    Found existing installation: huggingface_hub 1.11.0
    Uninstalling huggingface_hub-1.11.0:
      Successfully uninstalled huggingface_hub-1.11.0
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the fol

In [ ]:
import torch
from transformers import BertTokenizer, BertForSequenceClassification

print(torch.__version__)
print(torch.cuda.is_available())

2.10.0+cpu
False


In [ ]:
tokenizer = BertTokenizer.from_pretrained("klue/bert-base")

model = BertForSequenceClassification.from_pretrained(
    "klue/bert-base",
    num_labels=2
)

print("모델 로드 성공")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


모델 로드 성공


In [ ]:
import pandas as pd
import numpy as np
import urllib.request
import os
from tqdm import tqdm

In [ ]:
urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_train.txt",
    filename="ratings_train.txt"
)

urllib.request.urlretrieve(
    "https://raw.githubusercontent.com/e9t/nsmc/master/ratings_test.txt",
    filename="ratings_test.txt"
)

('ratings_test.txt', <http.client.HTTPMessage at 0x782daa3c2db0>)

In [ ]:
train_data = pd.read_table('ratings_train.txt')
test_data = pd.read_table('ratings_test.txt')

In [ ]:
print('훈련용 리뷰 개수 :', len(train_data))
print('테스트용 리뷰 개수 :', len(test_data))

훈련용 리뷰 개수 : 150000
테스트용 리뷰 개수 : 50000


In [ ]:
print(tokenizer.tokenize("보는내내 그대로 들어맞는 예측 카리스마 없는 악역"))

['보', '##는', '##내', '##내', '그대로', '들어맞', '##는', '예측', '카리스마', '없', '##는', '악역']


## = 앞 단어에 이어 붙는 조각

BERT는 문장을 통째로 안 보고 “단어 조각” 단위로 봄

In [ ]:
print(tokenizer.encode("보는내내 그대로 들어맞는 예측 카리스마 없는 악역"))

[2, 1160, 2259, 2369, 2369, 4311, 20657, 2259, 5501, 13132, 1415, 2259, 23713, 3]


2, 3은 원래 문장 단어가 아니라 BERT가 붙인 “특수 토큰”이야.

[CLS] = 문장 시작 표시

[SEP] = 문장 끝 표시

그래서 decode() 하면 다시 사람이 읽는 문장처럼 복원됨.

In [ ]:
print(tokenizer.decode(tokenizer.encode("보는내내 그대로 들어맞는 예측 카리스마 없는 악역")))

[CLS] 보는내내 그대로 들어맞는 예측 카리스마 없는 악역 [SEP]


In [ ]:
print(tokenizer.cls_token, ':', tokenizer.cls_token_id)
print(tokenizer.sep_token, ':' , tokenizer.sep_token_id)

[CLS] : 2
[SEP] : 3


패딩 토큰 확인하기

In [ ]:
print(tokenizer.pad_token, ':', tokenizer.pad_token_id)

[PAD] : 0


In [ ]:
max_seq_len = 128

encoded_result = tokenizer.encode("전율을 일으키는 영화. 다시 보고싶은 영화", max_length=max_seq_len, pad_to_max_length=True)

print(encoded_result)
print('길이 :', len(encoded_result))

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.


[2, 1537, 2534, 2069, 6572, 2259, 3771, 18, 3690, 4530, 2585, 2073, 3771, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
길이 : 128


/usr/local/lib/python3.12/dist-packages/transformers/tokenization_utils_base.py:2619: FutureWarning: The `pad_to_max_length` argument is deprecated and will be removed in a future version, use `padding=True` or `padding='longest'` to pad to the longest sequence in the batch, or use `padding='max_length'` to pad to a max length. In this case, you can give a specific length with `max_length` (e.g. `max_length=45`) or leave max_length to None to pad to the maximal input size of the model (e.g. 512 for Bert).
  warnings.warn(


In [ ]:
# 세그멘트 인코딩
print([0]*max_seq_len)

[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
# 어텐션 마스크 인코딩
valid_num = len(tokenizer.encode('전율을 일으키는 영화. 다시 보고싶은 영화'))
print(valid_num * [1] + (max_seq_len - valid_num) * [0])

[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


입력된 전체 데이터에 대해서 이 과정을 진행하는 함수 만들기


In [ ]:
def convert_examples_to_features(examples, labels, max_seq_len, tokenizer):

    input_ids, attention_masks, token_type_ids, data_labels = [], [], [], []

    for example, label in tqdm(zip(examples, labels), total=len(examples)):

        input_id = tokenizer.encode(
            example,
            max_length=max_seq_len,
            padding='max_length',
            truncation=True
        )

        padding_count = input_id.count(tokenizer.pad_token_id)

        attention_mask = [1] * (max_seq_len - padding_count) + [0] * padding_count

        token_type_id = [0] * max_seq_len

        input_ids.append(input_id)
        attention_masks.append(attention_mask)
        token_type_ids.append(token_type_id)
        data_labels.append(label)

    input_ids = np.array(input_ids, dtype=int)
    attention_masks = np.array(attention_masks, dtype=int)
    token_type_ids = np.array(token_type_ids, dtype=int)

    data_labels = np.asarray(data_labels, dtype=np.int32)

    return (input_ids, attention_masks, token_type_ids), data_labels

In [ ]:
max_seq_len = 128

# 'document'와 'label' 컬럼에 NaN 값이 있는 행을 제거합니다.
train_data.dropna(subset=['document', 'label'], inplace=True)
test_data.dropna(subset=['document', 'label'], inplace=True)

train_X, train_y = convert_examples_to_features(
    train_data['document'],
    train_data['label'],
    max_seq_len=max_seq_len,
    tokenizer=tokenizer
)

test_X, test_y = convert_examples_to_features(
    test_data['document'],
    test_data['label'],
    max_seq_len=max_seq_len,
    tokenizer=tokenizer
)

100%|██████████| 49997/49997 [00:13<00:00, 3650.43it/s]


샘플 뜯어보는 단계

In [ ]:
input_id = train_X[0][0]
attention_mask = train_X[1][0]
token_type_id = train_X[2][0]
label = train_y[0]

print('단어에 대한 정수 인코딩 :', input_id)
print('어텐션 마스크 :', attention_mask)
print('세그먼트 인코딩 :', token_type_id)
print('각 인코딩의 길이 :', len(input_id))
print('정수 인코딩 복원 :', tokenizer.decode(input_id))
print('레이블 :', label)

단어에 대한 정수 인코딩 : [   2 1376  831 2604   18   18 4229 9801 2075 2203 2182 4243    3    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0    0    0    0    0    0    0    0    0    0    0    0    0
    0    0]
어텐션 마스크 : [1 1 1 1 1 1 1 1 1 1 1 1 1 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0
 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0]
세그먼트 인코딩 : [0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 0 

train_X[0][0] :  첫 번째 리뷰의 숫자 인코딩

attention_mask : 진짜 단어(1) / 패딩(0) 표시

token_type_id : 문장 구분 정보

decode : 숫자를 다시 문장으로 복원

label :  긍정(1) / 부정(0)

즉 “BERT에게 들어가기 직전 데이터 상태 검사” 단계라고 보면 됨.

In [ ]:
!pip install huggingface_hub==0.20.3

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 330.1/330.1 kB 7.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 0.36.2
    Uninstalling huggingface_hub-0.36.2:
      Successfully uninstalled huggingface_hub-0.36.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 5.50.0 requires huggingface-hub<2.0,>=0.33.5, but you have huggingface-hub 0.20.3 which is incompatible.
diffusers 0.37.1 requires huggingface-hub<2.0,>=0.34.0, but you have huggingface-hub 0.20.3 which is incompatible.
datasets 4.0.0 requires huggingface-hub>=0.24.0, but you have huggingface-hub 0.20.3 which is incompatible.
peft 0.19.1 requires huggingface_hub>=0.25.0, but you have huggingface-hub 0.20.3 which is incompatible.
accelerate 1.13.0 requires huggingface_hub>=0.21.0, but you have huggingface-hub 0.20.3 which is incompatible.
sen

In [ ]:
from transformers import BertModel

In [ ]:
model = BertModel.from_pretrained("klue/bert-base")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(


In [ ]:
# outpus 변수에 저장하기, 입력문장길이 128로 가정하기

sample_input = tokenizer(
    "보는내내 그대로 들어맞는 예측 카리스마 없는 악역",
    return_tensors="pt",
    padding="max_length",
    truncation=True,
    max_length=128
)


In [ ]:
outputs = model(**sample_input)

In [ ]:
print(outputs.last_hidden_state.shape)

torch.Size([1, 128, 768])


1 : 문장 1개 넣음

128 : 문장 길이를 128칸으로 맞춤
(PAD 포함)

768 :  단어 하나를 설명하는 특징 숫자 768개

In [ ]:
cls_output = outputs.last_hidden_state[:, 0, :]
print(cls_output.shape)

torch.Size([1, 768])


In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

print("사용 장치:", device)

사용 장치: cpu


 Many-to-One 모델 만들기

데이터가 너무 많아 로딩이 오래걸려서 다 지우고 데이터 20000개 + batch_size 64 + epoch 1로 줄였습니다

In [ ]:
from transformers import BertForSequenceClassification
from torch.utils.data import TensorDataset, DataLoader, Subset
from torch.optim import AdamW
import torch.nn as nn
import random

In [ ]:
model = BertForSequenceClassification.from_pretrained(
    "klue/bert-base",
    num_labels=2
).to(device)

optimizer = AdamW(model.parameters(), lr=5e-5)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Some weights of BertForSequenceClassification were not initialized from the model checkpoint at klue/bert-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# 데이터준비

train_input_ids, train_attention_masks, train_token_type_ids = train_X
test_input_ids, test_attention_masks, test_token_type_ids = test_X

train_dataset = TensorDataset(
    torch.tensor(train_input_ids, dtype=torch.long),
    torch.tensor(train_attention_masks, dtype=torch.long),
    torch.tensor(train_token_type_ids, dtype=torch.long),
    torch.tensor(train_y, dtype=torch.long)
)

test_dataset = TensorDataset(
    torch.tensor(test_input_ids, dtype=torch.long),
    torch.tensor(test_attention_masks, dtype=torch.long),
    torch.tensor(test_token_type_ids, dtype=torch.long),
    torch.tensor(test_y, dtype=torch.long)
)

# 데이터 20000개로 샘플링
indices = random.sample(range(len(train_dataset)), 20000)
train_dataset_small = Subset(train_dataset, indices)

train_loader = DataLoader(train_dataset_small, batch_size=64, shuffle=True)
test_loader  = DataLoader(test_dataset, batch_size=64, shuffle=False)

print("학습 배치 수:", len(train_loader))  # 약 313

학습 배치 수: 313


In [ ]:
# 학습

model.train()

for epoch in range(1):
    total_loss = 0

    for i, (input_ids, attention_mask, token_type_ids, labels) in enumerate(train_loader):
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        token_type_ids = token_type_ids.to(device)
        labels         = labels.to(device)

        optimizer.zero_grad()
        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            labels=labels
        )
        outputs.loss.backward()
        optimizer.step()

        total_loss += outputs.loss.item()

        if i % 50 == 0:
            print(f"  batch {i}/{len(train_loader)} | loss: {outputs.loss.item():.4f}")

    print(f"✅ epoch 1 완료 | avg loss: {total_loss/len(train_loader):.4f}")

In [ ]:
model.eval()
correct = total = 0

with torch.no_grad():
    for input_ids, attention_mask, token_type_ids, labels in test_loader:
        input_ids      = input_ids.to(device)
        attention_mask = attention_mask.to(device)
        token_type_ids = token_type_ids.to(device)
        labels         = labels.to(device)

        outputs = model(input_ids=input_ids, attention_mask=attention_mask, token_type_ids=token_type_ids)
        preds   = torch.argmax(outputs.logits, dim=1)
        correct += (preds == labels).sum().item()
        total   += labels.size(0)

print(f"테스트 정확도: {correct/total*100:.2f}%")